In [1]:
import pandas as pd
from demandfcst import config, storage
from demandfcst.pipeline import schema

loc = config.locations()
history = storage.join(loc["landing"], "sellin_daily", "ingest_date=2024-12-29", "part.csv")
daily = schema.read_csv(history, schema.SCHEMAS["sellin_daily"])
print(daily.shape)
print(daily.dtypes)
daily.head(3)


(223125, 4)
date         datetime64[us]
province                str
sku_id                  str
volume_hl           float64
dtype: object


,date,province,sku_id,volume_hl
0,2021-01-01,BA,18747,2.100
1,2021-01-01,BA,22575,2.692
2,2021-01-01,BA,27988,4.774


In [2]:
skus = schema.read_csv(storage.join(loc["landing"], "sku_master", "ingest_date=2024-12-29", "part.csv"),
                       schema.SCHEMAS["sku_master"])
print(skus.dtypes)
skus[skus["launch_date"].notna() | skus["discontinue_date"].notna()][["sku_id", "sku_name", "launch_date", "discontinue_date"]]


sku_id                         str
sku_name                       str
brand                          str
pack                           str
container                      str
size_ml                      Int64
units_per_case               Int64
returnable                 boolean
launch_date         datetime64[us]
discontinue_date    datetime64[us]
dtype: object


,sku_id,sku_name,launch_date,discontinue_date
15,73000,BUDWEISER LATA 269ML,2023-06-05,NaT
16,32093,BUDWEISER LATA 350ML,2024-04-01,NaT
17,24456,BUDWEISER LATA 473ML,NaT,2022-12-25
24,51300,ORIGINAL LONG NECK 330ML,2022-03-07,NaT
27,90806,STELLA ARTOIS LATA 350ML,NaT,2023-08-27
35,35942,SPATEN LATA 269ML,2022-10-03,NaT
38,31554,SPATEN GARRAFA 600ML,NaT,2024-06-30


In [3]:
bad = pd.DataFrame({"date": ["2024-01-01", "01/02/2024"], "province": ["SP", None],
                    "sku_id": ["00123", "38958"], "volume_hl": ["1.5", "abc"]})
try:
    schema.coerce(bad, schema.SCHEMAS["sellin_daily"])
except schema.SchemaError as e:
    for p in e.problems:
        print("-", p)


- date: 1 value(s) are not date, e.g. ['01/02/2024']
- province: 1 empty value(s)
- volume_hl: 1 value(s) are not float, e.g. ['abc']


In [4]:
from demandfcst import config
from demandfcst.pipeline import reader

raw = config.pipeline_paths()["raw"]

sales = reader.read_dataset(raw, "sellin_daily")
print(sales.shape)
sales.groupby("ingest_date").agg(rows=("volume_hl", "size"),
                                 first_day=("date", "min"),
                                 last_day=("date", "max"),
                                 total_hl=("volume_hl", "sum")).round(0)


(232962, 5)


C:\Users\haris\AppData\Local\Temp\ipykernel_33288\1963499026.py:11: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  total_hl=("volume_hl", "sum")).round(0)


,rows,first_day,last_day,total_hl
ingest_date,,,,
2024-12-29,223125,2021-01-01,2024-12-28,8983295.0
2025-01-05,1095,2024-12-30,2025-01-04,59486.0
2025-01-12,1096,2025-01-06,2025-01-11,48505.0
2025-01-19,1087,2025-01-13,2025-01-18,56121.0
2025-01-26,1091,2025-01-20,2025-01-25,55093.0
2025-02-02,1091,2025-01-27,2025-02-01,56424.0
2025-02-09,1094,2025-02-03,2025-02-08,59362.0
2025-02-16,1087,2025-02-10,2025-02-15,54581.0
2025-02-23,1093,2025-02-17,2025-02-22,65312.0


In [5]:
wx = reader.read_dataset(raw, "weather_weekly")
skus = reader.read_latest(raw, "sku_master")
provinces = reader.read_latest(raw, "province_master")
print("weather:", wx.shape, "| skus:", skus.shape, "| provinces:", provinces.shape)
print("master snapshot from:", skus["ingest_date"].iloc[0].date())


weather: (1962, 7) | skus: (40, 11) | provinces: (9, 4)
master snapshot from: 2024-12-29


In [6]:
from demandfcst import config
from demandfcst.pipeline import reader, validate

raw = config.pipeline_paths()["raw"]
tables = reader.read_all(raw)
for name, df in tables.items():
    print(f"{name:16} {df.shape}")

report = validate.validate(tables)
print("no issues ✅" if report.empty else report)


sellin_daily     (232962, 5)
weather_weekly   (1962, 7)
province_master  (9, 4)
sku_master       (40, 11)
holidays         (260, 4)
     level       dataset          check  \
0  warning  sellin_daily  restated_rows   

                                             message  
0  32 rows restate a day delivered in another dro...  


In [7]:
broken = {name: df.copy() for name, df in tables.items()}
s = broken["sellin_daily"]
s = pd.concat([s, s.iloc[[0]]], ignore_index=True)        # the same row twice in one delivery
s.loc[1, "volume_hl"] = -5.0                                # a negative volume
s.loc[2, "sku_id"] = "99999"                                # a product nobody knows
broken["sellin_daily"] = s
broken["weather_weekly"] = broken["weather_weekly"].iloc[1:]   # one weather week lost

report = validate.validate(broken)
for r in report.itertuples():
    print(f"[{r.level}] {r.dataset}.{r.check}: {r.message[:110]}")


[error] sellin_daily.duplicate_in_drop: 2 rows share a key within one drop, e.g. [{'province': 'BA', 'sku_id': '18747', 'date': '2021-01-01', 'ingest_
[warning] sellin_daily.restated_rows: 32 rows restate a day delivered in another drop (latest drop wins), e.g. [{'province': 'MG', 'sku_id': '79082'
[error] sellin_daily.volume_not_positive: 1 rows with volume_hl <= 0, e.g. [{'province': 'BA', 'sku_id': '22575', 'date': '2021-01-01', 'volume_hl': '-5
[error] sellin_daily.unknown_sku: 1 rows with a sku_id not in sku_master, e.g. [{'province': 'BA', 'sku_id': '99999', 'date': '2021-01-01'}]
[error] weather_weekly.missing_weather_week: 1 province-weeks with no weather, e.g. [{'province': 'BA', 'week_start': '2020-12-28'}]
[error] sellin_daily.sales_without_weather: 1 province-weeks with sales but no weather, e.g. [{'province': 'BA', 'week_start': '2020-12-28'}]


In [8]:
import numpy as np
import pandas as pd
from demandfcst import config, drops, storage
from demandfcst.pipeline import dedupe, reader, validate

raw = config.pipeline_paths()["raw"]
tables = reader.read_all(raw)

report = validate.validate(tables)
print(report[["level", "check", "message"]].to_string(max_colwidth=100))

current, superseded = dedupe.latest_wins(tables["sellin_daily"], validate.SALES_KEY)
print(len(tables["sellin_daily"]), "rows read ->", len(current), "current +", len(superseded), "superseded")
dedupe.restatement_summary(current, superseded, validate.SALES_KEY)


     level          check                                                                                              message
0  warning  restated_rows  32 rows restate a day delivered in another drop (latest drop wins), e.g. [{'province': 'MG', 'sk...
232962 rows read -> 232946 current + 16 superseded


,ingest_date,restated_days,hl_before,hl_after,hl_change
0,2025-03-02,16,615.814,616.115,0.301


In [9]:
truth_root = config.locations()["truth"]
files = [storage.join(drops.drop_dir(truth_root, "corrections", d), "part.csv")
         for d in drops.drop_dates(truth_root, "corrections")]
answer = pd.concat([pd.read_csv(f, dtype={"province": str, "sku_id": str}, parse_dates=["date"]) for f in files],
                   ignore_index=True)
check = answer.merge(current, on=["date", "province", "sku_id"], suffixes=("_truth", "_pipeline"))
print(len(answer), "corrections sent |", len(check), "found |",
      "all applied ✅" if np.allclose(check["volume_hl_truth"], check["volume_hl_pipeline"]) else "MISMATCH ❌")


ValueError: You are trying to merge on object and datetime64[us] columns for key 'date'. If you wish to proceed you should use pd.concat

In [10]:
from demandfcst import config, drops
from demandfcst.pipeline import dedupe, reader, validate, weekly

raw = config.pipeline_paths()["raw"]
tables = reader.read_all(raw)
current, _ = dedupe.latest_wins(tables["sellin_daily"], validate.SALES_KEY)

start = config.sales_start()
end = drops.drop_dates(raw, "sellin_daily")[-1]
wk = weekly.to_weekly(current, tables["sku_master"], start, end)

print(wk.shape)
print("weeks:", wk["week_start"].min().date(), "→", wk["week_start"].max().date(), "|", wk["week_start"].nunique())
print("series:", wk.groupby(["province", "sku_id"]).ngroups, "| zero weeks:", (wk["volume_hl"] == 0).sum())
wk.head()


(46576, 4)
weeks: 2021-01-04 → 2025-02-24 | 217
series: 231 | zero weeks: 4168


,province,sku_id,week_start,volume_hl
0,BA,10662,2021-01-25,68.658
1,BA,10662,2021-02-01,0.000
2,BA,10662,2021-02-08,71.888
3,BA,10662,2021-02-15,0.000
4,BA,10662,2021-02-22,65.470


In [11]:
seg_check = wk.groupby(["province", "sku_id"])["volume_hl"].agg(zero_share=lambda s: (s == 0).mean())
p, s = seg_check["zero_share"].idxmax()                       # the sparsest series
wk[(wk["province"] == p) & (wk["sku_id"] == s)].tail(8)


,province,sku_id,week_start,volume_hl
14327,PE,44863,2025-01-06,27.986
14328,PE,44863,2025-01-13,0.000
14329,PE,44863,2025-01-20,0.000
14330,PE,44863,2025-01-27,0.000
14331,PE,44863,2025-02-03,0.000
14332,PE,44863,2025-02-10,0.000
14333,PE,44863,2025-02-17,0.000
14334,PE,44863,2025-02-24,69.084


In [12]:
disc = tables["sku_master"].dropna(subset=["discontinue_date"])[["sku_id", "sku_name", "discontinue_date"]]
print(disc)
last_weeks = wk[wk["sku_id"].isin(disc["sku_id"])].groupby("sku_id")["week_start"].max()
print(last_weeks)


   sku_id                  sku_name discontinue_date
17  24456      BUDWEISER LATA 473ML       2022-12-25
27  90806  STELLA ARTOIS LATA 350ML       2023-08-27
38  31554      SPATEN GARRAFA 600ML       2024-06-30
sku_id
24456   2022-12-19
31554   2024-06-24
90806   2023-08-21
Name: week_start, dtype: datetime64[us]


In [1]:
import pandas as pd
from demandfcst import config

sw = pd.read_parquet(config.pipeline_paths()["processed"] + "/sales_weekly.parquet")
print(sw.dtypes)
print(sw.head(3))


province                 str
sku_id                   str
week_start    datetime64[us]
volume_hl            float64
dtype: object
  province sku_id week_start  volume_hl
0       BA  10662 2021-01-25     68.658
1       BA  10662 2021-02-01      0.000
2       BA  10662 2021-02-08     71.888
